## Import và cấu hình
Ở đây em dùng dataset UIT-ViQuAD 2.0

In [4]:
from datasets import load_dataset
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
import os

DATASET_NAME = "taidng/UIT-ViQuAD2.0"
SPLIT = "train[:300]"
EMBED_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
VECTOR_DB_PATH = "./qdrant_viquad"
COLLECTION_NAME = "viquad_demo"
TOP_K = 3

## Load dataset

In [7]:
raw_data = load_dataset(DATASET_NAME, split=SPLIT)
print(raw_data[0])

{'id': '0001-0001-0001', 'uit_id': 'uit_000001', 'title': 'Phạm Văn Đồng', 'context': 'Phạm Văn Đồng (1 tháng 3 năm 1906 – 29 tháng 4 năm 2000) là Thủ tướng đầu tiên của nước Cộng hòa Xã hội chủ nghĩa Việt Nam từ năm 1976 (từ năm 1981 gọi là Chủ tịch Hội đồng Bộ trưởng) cho đến khi nghỉ hưu năm 1987. Trước đó ông từng giữ chức vụ Thủ tướng Chính phủ Việt Nam Dân chủ Cộng hòa từ năm 1955 đến năm 1976. Ông là vị Thủ tướng Việt Nam tại vị lâu nhất (1955–1987). Ông là học trò, cộng sự của Chủ tịch Hồ Chí Minh. Ông có tên gọi thân mật là Tô, đây từng là bí danh của ông. Ông còn có tên gọi là Lâm Bá Kiệt khi làm Phó chủ nhiệm cơ quan Biện sự xứ tại Quế Lâm (Chủ nhiệm là Hồ Học Lãm).', 'question': 'Tên gọi nào được Phạm Văn Đồng sử dụng khi làm Phó chủ nhiệm cơ quan Biện sự xứ tại Quế Lâm?', 'answers': {'text': ['Lâm Bá Kiệt'], 'answer_start': [507]}, 'is_impossible': False, 'plausible_answers': None}


## Tạo documents cho RAG
Mỗi context sẽ trở thành một document.
Ta loại bỏ context trùng nhau để vector store gọn hơn.

In [8]:
seen_contexts = set()
documents = []

for item in raw_data:
    context = item["context"].strip()
    title = item.get("title", "")

    if context and context not in seen_contexts:
        seen_contexts.add(context)
        documents.append(
            Document(
                page_content=context,
                metadata={"title": title}
            )
        )

print(f"Số document sau khi bỏ trùng: {len(documents)}")
print(documents[0].metadata)
print(documents[0].page_content[:500])

Số document sau khi bỏ trùng: 45
{'title': 'Phạm Văn Đồng'}
Phạm Văn Đồng (1 tháng 3 năm 1906 – 29 tháng 4 năm 2000) là Thủ tướng đầu tiên của nước Cộng hòa Xã hội chủ nghĩa Việt Nam từ năm 1976 (từ năm 1981 gọi là Chủ tịch Hội đồng Bộ trưởng) cho đến khi nghỉ hưu năm 1987. Trước đó ông từng giữ chức vụ Thủ tướng Chính phủ Việt Nam Dân chủ Cộng hòa từ năm 1955 đến năm 1976. Ông là vị Thủ tướng Việt Nam tại vị lâu nhất (1955–1987). Ông là học trò, cộng sự của Chủ tịch Hồ Chí Minh. Ông có tên gọi thân mật là Tô, đây từng là bí danh của ông. Ông còn có tên 


## Chia nhỏ văn bản
Chia chunk để truy xuất ổn định hơn.

In [9]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = text_splitter.split_documents(documents)
print(f"Tổng số chunks: {len(chunks)}")
print(chunks[0].page_content[:300])

Tổng số chunks: 127
Phạm Văn Đồng (1 tháng 3 năm 1906 – 29 tháng 4 năm 2000) là Thủ tướng đầu tiên của nước Cộng hòa Xã hội chủ nghĩa Việt Nam từ năm 1976 (từ năm 1981 gọi là Chủ tịch Hội đồng Bộ trưởng) cho đến khi nghỉ hưu năm 1987. Trước đó ông từng giữ chức vụ Thủ tướng Chính phủ Việt Nam Dân chủ Cộng hòa từ năm 19


## Tạo embeddings và vector store

In [10]:
import shutil

shutil.rmtree(VECTOR_DB_PATH, ignore_errors=True)

embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

vectorstore = QdrantVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    path=VECTOR_DB_PATH,
    collection_name=COLLECTION_NAME,
)

retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": TOP_K, "fetch_k": 10}
)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## Demo một câu query

In [19]:
sample = raw_data[1]
question = sample["question"]

answer_list = sample["answers"]["text"]
gold_answer = answer_list[0] if len(answer_list) > 0 else "Không có đáp án"

retrieved_docs = retriever.invoke(question)

print("Câu hỏi mẫu:", question)
print("Đáp án tham chiếu:", gold_answer)
print("\nTop documents retrieve được:\n")

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"--- Document {i} ---")
    print("Title:", doc.metadata.get("title", ""))
    print(doc.page_content[:400])
    print()

Câu hỏi mẫu: Phạm Văn Đồng giữ chức vụ gì trong bộ máy Nhà nước Cộng hòa Xã hội chủ nghĩa Việt Nam?
Đáp án tham chiếu: Thủ tướng

Top documents retrieve được:

--- Document 1 ---
Title: Phạm Văn Đồng
Phạm Văn Đồng (1 tháng 3 năm 1906 – 29 tháng 4 năm 2000) là Thủ tướng đầu tiên của nước Cộng hòa Xã hội chủ nghĩa Việt Nam từ năm 1976 (từ năm 1981 gọi là Chủ tịch Hội đồng Bộ trưởng) cho đến khi nghỉ hưu năm 1987. Trước đó ông từng giữ chức vụ Thủ tướng Chính phủ Việt Nam Dân chủ Cộng hòa từ năm 1955 đến năm 1976. Ông là vị Thủ tướng Việt Nam tại vị lâu nhất (1955–1987). Ông là học trò, cộng sự c

--- Document 2 ---
Title: Phạm Văn Đồng
Năm 1954, ông được giao nhiệm vụ Trưởng phái đoàn Chính phủ dự Hội nghị Genève về Đông Dương. Những đóng góp của đoàn Việt Nam do ông đứng đầu là vô cùng quan trọng, tạo ra những đột phá đưa Hội nghị tới thành công

--- Document 3 ---
Title: Phạm Văn Đồng
Ông Việt Phương, nguyên thư ký của Thủ tướng Phạm Văn Đồng, trong buổi họp báo giới thiệu sách của các